In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

In [2]:

print("Loading dataset for Classifier Training...")
df = pd.read_csv("smart_traffic_management_bengaluru_14days_30min_CORRECTED.csv")


Loading dataset for Classifier Training...


In [3]:


df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.hour / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.hour / 24.0)



df['max_capacity'] = 250 
df['lane_occupancy'] = np.minimum(100, (df['number_of_vehicles'] / df['max_capacity']) * 100)


In [ ]:
np.random.seed(42)
occupancy_noise = np.random.uniform(-12, 12, size=len(df))
effective_occupancy = df['lane_occupancy'] + occupancy_noise
adv_conditions = [
    (effective_occupancy > 80) & (df['weather_condition'] != 'Sunny'),
    (effective_occupancy > 80),
    (effective_occupancy > 60)
]
df['ai_advisory'] = np.select(adv_conditions, ["Green Wave Sync", "+25s Green Phase", "Adaptive Phase"], default="Monitor Cycle")

inc_conditions = [
    (df['accident_reported'] == 1) & (effective_occupancy > 75),
    (df['accident_reported'] == 1) & (effective_occupancy <= 75)
]
df['incident_action'] = np.select(inc_conditions, ["Dispatch Quick Response & Tow", "Assign Patrol Unit"], default="No Action")

categorical_vars = ['weather_condition']

In [8]:
dispatch_prob = np.random.rand(len(df))

inc_conditions = [
  
    (df['accident_reported'] == 1) & (effective_occupancy > 75) & (dispatch_prob > 0.15),
    (df['accident_reported'] == 1) & (effective_occupancy > 75) & (dispatch_prob <= 0.15),
    
    
    (df['accident_reported'] == 1) & (effective_occupancy <= 75) & (dispatch_prob > 0.20),
    (df['accident_reported'] == 1) & (effective_occupancy <= 75) & (dispatch_prob <= 0.20),
    
    
    (df['accident_reported'] == 0) & (df['lane_occupancy'] > 85) & (dispatch_prob < 0.10),
    
    (df['accident_reported'] == 0) & (df['weather_condition'].isin(['Rainy', 'Foggy'])) & (dispatch_prob < 0.05)
]

inc_choices = [
    "Dispatch Quick Response & Tow", # 1
    "Assign Patrol Unit",            # 1
    "Assign Patrol Unit",            # 2
    "Dispatch Quick Response & Tow", # 2
    "Assign Patrol Unit",            # 3
    "Assign Patrol Unit"             # 4
]

df['incident_action'] = np.select(inc_conditions, inc_choices, default="No Action")
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [6]:


print("Training Hotspot Advisory Classifier...")
hotspot_features = ['hour_sin', 'hour_cos', 'weather_condition', 'number_of_vehicles', 'lane_occupancy']
cat_hotspot = CatBoostClassifier(iterations=300, depth=6, random_seed=42, verbose=False)
cat_hotspot.fit(train_df[hotspot_features], train_df['ai_advisory'], cat_features=categorical_vars)
hotspot_preds = cat_hotspot.predict(test_df[hotspot_features])
print(f"2. Hotspot Advisory Model (Classifier) -> Accuracy: {accuracy_score(test_df['ai_advisory'], hotspot_preds)*100:.2f}%")
cat_hotspot.fit(df[hotspot_features], df['ai_advisory'], cat_features=categorical_vars)
cat_hotspot.save_model("hotspot_model.cbm")

Training Hotspot Advisory Classifier...
2. Hotspot Advisory Model (Classifier) -> Accuracy: 89.83%


In [9]:


print("Training Incident Action Classifier...")
incident_features = ['weather_condition', 'number_of_vehicles', 'lane_occupancy', 'accident_reported']
cat_incident = CatBoostClassifier(iterations=300, depth=6, random_seed=42, verbose=False)
cat_incident.fit(train_df[incident_features], train_df['incident_action'], cat_features=categorical_vars)
inc_preds = cat_incident.predict(test_df[incident_features])
print(f"3. Incident Action Model (Classifier) -> Accuracy: {accuracy_score(test_df['incident_action'], inc_preds)*100:.2f}%")
cat_incident.fit(df[incident_features], df['incident_action'], cat_features=categorical_vars)
cat_incident.save_model("incident_model.cbm")

Training Incident Action Classifier...
3. Incident Action Model (Classifier) -> Accuracy: 95.02%
